# HBI tutorials — NB0: motivation & the estimand

**Audience.** A new collaborator who will *run* the catalog-HBI CDDF pipeline, and a
PI doing a math review. This notebook sets up *what we are measuring* and *why a
naive feed-forward count gets it wrong*. It runs in a few seconds and re-runs no
inference.

## The 5-notebook series

| NB | Title | What it does |
|----|-------|--------------|
| **00** | **motivation & the estimand** *(this one)* | Defines $f(N,X)$, $dN/dX$, $\Omega_\mathrm{DLA}$; shows *why* $\Omega$ is tail-dominated; previews the destination figure. |
| 01 | synthetic end-to-end | Injects a known CDDF into a toy catalog, forward-simulates detections, and recovers it — the whole machine in one runnable file. |
| 02 | forward kernel | Builds the per-object $(N,z)$ forward response (the GP posterior up-migration near the prior edge) that HBI inverts. |
| 03 | likelihood + MAP + band | The marked-Poisson likelihood, convex MAP, and the joint-MC credible band. |
| 04 | validation | Cross-mock transfer — the genuine external test (on-mock recovery is self-consistency, see the last cell). |

## The reduce-only discipline

These notebooks are **reduce-only**: GP inference is **never re-run** here. The
pipeline consumes a *frozen* detection catalog (the GP-DLA output) plus a frozen
completeness/purity matrix, and reduces them to a population statistic. Nothing in
this series touches `dla_gp.py` or re-runs Bayesian model selection. That keeps the
expensive, NERSC/GreatLakes-proven inference path byte-identical and makes every
tutorial cell cheap and deterministic.

> **Data provenance.** Every printed number below comes from running real code or
> from a committed **MOCK** fixture (2LPT-0 injection, truth known). There are **no
> real-survey values** anywhere in this notebook.

In [1]:
# --- deterministic environment: set BEFORE importing numpy / CDDF ---
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS",
           "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = "1"

import sys
from pathlib import Path

# Render matplotlib figures inline in the notebook.
get_ipython().run_line_magic("matplotlib", "inline")

# Resolve the repo root by walking up until we find the CDDF_analysis package.
repo_root = Path.cwd()
while not (repo_root / "CDDF_analysis").is_dir():
    if repo_root == repo_root.parent:
        raise RuntimeError("could not locate CDDF_analysis above %s" % Path.cwd())
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
print("repo_root:", repo_root)

import json
import numpy as np
import matplotlib.pyplot as plt

# Canonical import path for the estimator.
from CDDF_analysis.hbi import cddf_catalog_hbi as H
from CDDF_analysis import cddf_mock

print("numpy:", np.__version__)
print("H module:", H.__name__)

repo_root: /tmp/claude-114399728/-home-mfho-desi-gpy-dla-detection/61bff2ab-596e-490e-9db3-2dbe8f55d4d9/scratchpad/hbi_tutorial_wt


numpy: 2.4.4
H module: CDDF_analysis.hbi.cddf_catalog_hbi


**Import path note.** The canonical path is
`from CDDF_analysis.hbi import cddf_catalog_hbi as H`. The older
`CDDF_analysis.cddf_catalog_hbi` is a thin **back-compat shim** that re-exports the
same symbols — new code should import from `CDDF_analysis.hbi`.

## The estimand

The **column-density distribution function** (CDDF) $f(N, X)$ is the number of
absorbers per unit column density $N \equiv N_\mathrm{HI}$ (in $\mathrm{cm}^{-2}$)
per unit absorption distance $X$:

$$ f(N, X) \;=\; \frac{\partial^2 \mathcal{N}}{\partial N \,\partial X}. $$

We work on a fine $\log N$ grid of bins $b$ with heights $f_b$. Two integral
quantities are reported.

**Line density above a threshold** $N_\mathrm{thr}$:

$$ \frac{dN}{dX}(\ge N_\mathrm{thr}) \;=\; \sum_{b\,:\,N_b \ge N_\mathrm{thr}} f_b \,\Delta N_b. $$

**Neutral-gas mass density:**

$$ \Omega_\mathrm{DLA}(\ge N_\mathrm{thr}) \;=\; K \sum_{b\,:\,N_b \ge N_\mathrm{thr}} N_b \, f_b \,\Delta N_b,
   \qquad K = \frac{H_0\, m_\mathrm{H}}{c\,\rho_{c,0}} . $$

### Gotcha: $\Delta N_b$ is a **linear** width

The bin width that multiplies $f_b$ is

$$ \Delta N_b \;=\; 10^{\log N_{\mathrm{hi},b}} - 10^{\log N_{\mathrm{lo},b}}, $$

a **linear** width in $N$, **not** $\Delta \log N$ and **not** $N\ln 10$. Because
the grid is uniform in $\log N$, $\Delta N_b$ grows by a factor $\sim 10$ per dex —
the high-$N$ bins are *enormously* wide in linear $N$.

### Consequence: $\Omega$ is tail-dominated

$dN/dX$ weights each bin by $f_b\,\Delta N_b$, but $\Omega$ weights by
$N_b\, f_b\,\Delta N_b$ — an *extra* factor of $N_b$. With $f(N)\propto N^{-\beta}$,
$\beta\!\sim\!1.5\text{–}2$, the integrand $N f$ falls only slowly, so the highest-$N$
bins carry a large share of $\Omega$. A small error in the *shape* of the high-$N$
tail barely moves $dN/dX$ but can move $\Omega$ markedly. We demonstrate this
mechanism numerically next.

In [2]:
# Build a fine logN grid via the REAL estimator helper.
# A minimal HBIConfig (paths are dummies; we never load them here) — mirrors
# the _make_cfg(...) helper in tests/test_cddf_catalog_hbi.py, but restricted to a
# clean 20.0-22.0 DLA window so the toy power law is easy to read.
cfg = H.HBIConfig(
    catalog_dir="/dev/null", truth_path="/dev/null",
    bal_cat_path="/dev/null", molly_tsv="/dev/null", out_dir="/tmp",
    logN_lo=20.0, logN_hi=22.0, dlogN=0.1, drop_top_bin_above=21.9,
    zbins=(2.0, 2.5, 3.0, 3.5), report_logN_limits=(20.0, 20.3),
    H0=70.0, Omega_m=0.279,
)
logN_lo, logN_hi, N_b, dN_b = H.build_fine_grid(cfg)
print("fine grid: %d bins, logN in [%.2f, %.2f]" % (len(N_b), logN_lo[0], logN_hi[-1]))

# --- SELF-CHECK: dN_b is the LINEAR width 10^hi - 10^lo (gotcha 2) ---
expected = 10.0 ** logN_hi - 10.0 ** logN_lo
assert np.allclose(dN_b, expected, rtol=1e-12)
# and explicitly NOT the common N*ln10 mistake:
assert not np.allclose(dN_b, N_b * np.log(10.0), rtol=1e-3)
print("PASS: dN_b is linear (10^hi - 10^lo), not N*ln10")
print("  e.g. bin @ logN~%.2f: dN_b = %.3e cm^-2  (N_b*ln10 would be %.3e)"
      % (np.log10(N_b[-1]), dN_b[-1], N_b[-1] * np.log(10.0)))

# --- A toy power-law CDDF f_b = A * N_b^beta, anchored at logN = 20.3 ---
# beta ~ -2.5 is representative of the DLA high-N tail (it steepens above the
# break); steep enough that the high-N bins carry few absorbers (small dN/dX
# share) but still a large share of Omega (extra N_b weight).
beta = -2.5
A = 1e-22 / (10 ** 20.3) ** beta          # height ~1e-22 cm^2 at N = 10^20.3
f_b = A * N_b ** beta

# dN/dX(>=20.3): sum over bins with N_b >= 10^20.3
sel = N_b >= 10 ** 20.3
dndx_303 = np.sum(f_b[sel] * dN_b[sel])
print("\ntoy power law beta=%.1f" % beta)
print("dN/dX(>=20.3) = %.4f" % dndx_303)
# Omega needs K -> computed with the real prefactor in cell 7; finish there.

fine grid: 19 bins, logN in [20.00, 21.90]
PASS: dN_b is linear (10^hi - 10^lo), not N*ln10
  e.g. bin @ logN~21.85: dN_b = 1.634e+21 cm^-2  (N_b*ln10 would be 1.630e+22)

toy power law beta=-2.5
dN/dX(>=20.3) = 0.0132


In [3]:
# Perturb the toy CDDF the way the raw feed-forward error does: a too-flat high-N
# tail = the heights above logN = 21.0 are over-stated. Here a MODEST 30% boost of
# the tail heights (a realistic magnitude for posterior up-migration near the prior
# edge -- it makes the steep tail shallower).
boost = 1.30
flat_above = 21.0
f_b_pert = f_b.copy()
tail = np.log10(N_b) > flat_above
f_b_pert[tail] = f_b[tail] * boost

# dN/dX is insensitive to the tail (those bins carry few absorbers despite being
# linearly wide -- the steep f_b wins):
dndx_orig = np.sum(f_b[sel]      * dN_b[sel])
dndx_pert = np.sum(f_b_pert[sel] * dN_b[sel])
print("dN/dX(>=20.3):  original %.4f  ->  tail-boosted %.4f   (%+.1f%%)"
      % (dndx_orig, dndx_pert, 100 * (dndx_pert / dndx_orig - 1)))

# Omega weights by an EXTRA factor of N_b, so the same tail boost matters far more.
# Finish the numbers (and narration) once K is in hand in cell 7 -- carry the
# integrands (Omega/K) so we can scale by the real K there:
omega_integrand_orig = np.sum(N_b[sel] * f_b[sel]      * dN_b[sel])  # = Omega/K
omega_integrand_pert = np.sum(N_b[sel] * f_b_pert[sel] * dN_b[sel])
print("Omega integrand (Omega/K):  original %.3e  ->  tail-boosted %.3e   (%+.1f%%)"
      % (omega_integrand_orig, omega_integrand_pert,
         100 * (omega_integrand_pert / omega_integrand_orig - 1)))
print("\n--> The SAME 30%% tail boost moves dN/dX by a few %%, but Omega by several x")
print("    more. The extra N_b weight makes a too-flat high-N tail OVER-STATE Omega")
print("    -- this is the mechanism behind the raw feed-forward Omega over-statement.")

dN/dX(>=20.3):  original 0.0132  ->  tail-boosted 0.0136   (+2.6%)
Omega integrand (Omega/K):  original 6.711e+18  ->  tail-boosted 7.401e+18   (+10.3%)

--> The SAME 30%% tail boost moves dN/dX by a few %%, but Omega by several x
    more. The extra N_b weight makes a too-flat high-N tail OVER-STATE Omega
    -- this is the mechanism behind the raw feed-forward Omega over-statement.


<details>
<summary><b>The mass-density prefactor</b> $K$ (click to expand the math)</summary>

The neutral-gas mass density follows from converting a column-density integral into
a mean comoving mass density divided by the critical density:

$$ K = \frac{H_0\, m_\mathrm{H}}{c\,\rho_{c,0}}, \qquad
   \rho_{c,0} = \frac{3 H_0^2}{8\pi G}. $$

Substituting $\rho_{c,0}$,

$$ K = \frac{H_0\, m_\mathrm{H}}{c}\cdot\frac{8\pi G}{3 H_0^2}
     = \frac{8\pi G\, m_\mathrm{H}}{3\,c\,H_0}. $$

With $N$ in $\mathrm{cm}^{-2}$ and $f(N)$ in $\mathrm{cm}^{2}$, the product
$N f(N)\,dN$ is dimensionless and **$K$ is dimensionless**. For $H_0 = 70\ \mathrm{km\,s^{-1}\,Mpc^{-1}}$,
$K \approx 1.38\times10^{-23}$.

- **$h$-scaling:** $K \propto 1/H_0$, so $\Omega_\mathrm{DLA}\propto h^{-1}$. Always
  state the assumed $H_0$ alongside an $\Omega$.
- **Mass convention:** using the H-atom mass $m_\mathrm{H}$ gives the *neutral-hydrogen-only*
  $\Omega_\mathrm{HI}$ (Noterdaeme+2012 / Crighton+2015). For total neutral *gas*
  including helium ($\Omega_\mathrm{DLA}$), multiply by $\mu = 1/X_\mathrm{H}\approx1.3$.

</details>

In [4]:
# Call the REAL prefactor (from cddf_mock, re-exported on H). Signature:
#   omega_hi_prefactor(H0_km_s_Mpc=70.0) -> dimensionless K
K = H.omega_hi_prefactor(70.0)
print("K = %.6e  (dimensionless)" % K)

# --- SELF-CHECK: K matches the corrected value 1.376e-23 (NOT ~2.8e-28) ---
assert abs(K - 1.376e-23) < 1e-2 * 1.376e-23
print("PASS: K within 1%% of 1.376e-23")

# Now finish the Omega numbers from cells 4-5 with the real K.
omega_303      = K * omega_integrand_orig   # toy power-law Omega(>=20.3)
omega_303_pert = K * omega_integrand_pert   # tail-boosted Omega(>=20.3)
print("\ntoy power law, Omega(>=20.3):")
print("  original     Omega = %.3e" % omega_303)
print("  tail-boosted Omega = %.3e   (%+.1f%%)"
      % (omega_303_pert, 100 * (omega_303_pert / omega_303 - 1)))
print("\nThe same tail boost barely moved dN/dX but moved Omega several x more ->")
print("dN/dX and Omega respond DIFFERENTLY to a high-N tail-shape error, exactly")
print("because Omega carries the extra N_b weight.")

K = 1.375931e-23  (dimensionless)
PASS: K within 1%% of 1.376e-23

toy power law, Omega(>=20.3):
  original     Omega = 9.234e-05
  tail-boosted Omega = 1.018e-04   (+10.3%)

The same tail boost barely moved dN/dX but moved Omega several x more ->
dN/dX and Omega respond DIFFERENTLY to a high-N tail-shape error, exactly
because Omega carries the extra N_b weight.


In [5]:
# Load the committed MOCK fixture (2LPT-0 injection; truth known).
fix_path = repo_root / "CDDF_analysis" / "hbi" / "tutorial_data" / "compare_synthesis.json"
d = json.load(open(fix_path))
truth = d["table"]["truth"]
m = d["table"]["methods"]

# --- SELF-CHECK: the fixture reproduces the known qualitative behaviour ---
assert m["raw_feedforward"]["omega"]["20.3"]["R0"] > 1.2          # raw-FF over-states Omega
assert 1.0 <= m["HBI_purity_mixture"]["dndx"]["20.3"]["R0"] <= 1.2  # HBI brings dN/dX into band
print("PASS: raw-FF Omega R0 = %.3f (>1.2);  HBI_purity_mixture dN/dX R0 = %.3f (in [1.0,1.2])"
      % (m["raw_feedforward"]["omega"]["20.3"]["R0"],
         m["HBI_purity_mixture"]["dndx"]["20.3"]["R0"]))

thr = "20.3"
methods = ["raw_feedforward", "HBI_purity_mixture", "HBI_loa0"]
labels  = ["raw feed-forward", "HBI (purity-mix)", "HBI (loa0)"]
colors  = ["#c44", "#27a", "#2a7"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Panel A: dN/dX(>=20.3)
ax = axes[0]
ax.axhline(truth["dndx"][thr], color="k", ls="--", lw=1.4, label="truth (mock)")
for j, (key, lab, c) in enumerate(zip(methods, labels, colors)):
    rec = m[key]["dndx"][thr]
    ax.scatter([j], [rec["value"]], s=70, color=c, zorder=3, label=lab)
    if "ci68" in rec:
        lo, hi = rec["ci68"]
        ax.plot([j, j], [lo, hi], color=c, lw=2.5, zorder=2)
    ax.annotate("R0=%.3f" % rec["R0"], (j, rec["value"]),
                textcoords="offset points", xytext=(8, 0), fontsize=9, color=c)
ax.set_xticks(range(len(methods))); ax.set_xticklabels(labels, rotation=12, fontsize=8)
ax.set_ylabel(r"$dN/dX\,(\geq 20.3)$"); ax.set_title(r"line density $dN/dX$")
ax.legend(fontsize=8, loc="lower right")

# Panel B: 10^3 * Omega(>=20.3)
ax = axes[1]
ax.axhline(1e3 * truth["omega"][thr], color="k", ls="--", lw=1.4, label="truth (mock)")
for j, (key, lab, c) in enumerate(zip(methods, labels, colors)):
    rec = m[key]["omega"][thr]
    ax.scatter([j], [1e3 * rec["value"]], s=70, color=c, zorder=3, label=lab)
    if "ci68" in rec:
        lo, hi = rec["ci68"]
        ax.plot([j, j], [1e3 * lo, 1e3 * hi], color=c, lw=2.5, zorder=2)
    ax.annotate("R0=%.3f" % rec["R0"], (j, 1e3 * rec["value"]),
                textcoords="offset points", xytext=(8, 0), fontsize=9, color=c)
ax.set_xticks(range(len(methods))); ax.set_xticklabels(labels, rotation=12, fontsize=8)
ax.set_ylabel(r"$10^3\,\Omega_\mathrm{DLA}\,(\geq 20.3)$"); ax.set_title(r"mass density $\Omega$")
ax.legend(fontsize=8, loc="upper left")

fig.suptitle("The destination: HBI vs raw feed-forward at logN >= 20.3 "
             "(2LPT-0 MOCK injection, truth known)", fontsize=11)
fig.text(0.5, -0.04,
         "2LPT-0 MOCK injection (truth known); raw-FF under-counts dN/dX & over-states "
         "Omega; HBI deconvolution fixes Omega. No real-survey values.\n"
         "R0 = method / truth; R0=1 is perfect recovery.",
         ha="center", fontsize=8.5)
fig.tight_layout()
plt.show()

PASS: raw-FF Omega R0 = 1.468 (>1.2);  HBI_purity_mixture dN/dX R0 = 1.090 (in [1.0,1.2])


/tmp/ipykernel_2908823/4227877915.py:59: UserWarning: FigureCanvasPdf is non-interactive, and thus cannot be shown
  plt.show()


## What the next notebooks do

The figure above is the **destination** — the machine the rest of the series builds.
Two things to take away:

- **Raw feed-forward** under-counts $dN/dX$ (incompleteness) *and* over-states
  $\Omega$. The over-statement is the tail mechanism from cells 5/7: the GP posterior
  **up-migrates** column densities near the sharp $\log N \approx 20.3$ prior edge,
  flattening the high-$N$ tail, and the extra $N_b$ weight in $\Omega$ amplifies that
  shape error.
- **HBI** builds the per-object $(N,z)$ *forward response* and **inverts** it
  (deconvolution), bringing both $dN/dX$ and $\Omega$ back into agreement with the
  injected mock truth.

**NB1** builds and inverts that forward response on **synthetic** data you can run in
seconds — no GP, no catalog I/O — so you can see the whole inversion end-to-end
before meeting the real kernel (NB2) and likelihood (NB3).

> **Honesty.** The on-mock recovery shown here is a **self-consistency** check: the
> completeness correction is calibrated on the *same* mock, so the recovery factor
> $\alpha = 1/R_0$ is a tautology on that mock. The genuine **external** test is
> **cross-mock transfer** — calibrate on one mock, validate on a held-out mock built
> with a different recipe — which is the subject of **NB4**.